# Constant

In [1]:
DATA_DIR = "dataset/"
MODEL_DIR = "model/"
CHECKPOINT_DIR = "epoch_model/"
RESULT_DIR = "result/"

In [2]:
TRAIN_EN = DATA_DIR + "train_en.csv"
TRAIN_TCN = DATA_DIR + "train_tcn.csv"

TEST_EN = DATA_DIR + "dev_en.csv"
TEST_TCN = DATA_DIR + "dev_tcn.csv"

PREDICT_TCN = DATA_DIR + "test_tcn.csv"
OUTPUT_CSV = RESULT_DIR + "result.csv"

# Import Data

In [3]:
import pandas


raw_train_en = pandas.read_csv(TRAIN_EN)
raw_train_tcn = pandas.read_csv(TRAIN_TCN)

raw_test_en = pandas.read_csv(TEST_EN)
raw_test_tcn = pandas.read_csv(TEST_TCN)

raw_predict_tcn = pandas.read_csv(PREDICT_TCN)

In [4]:
train_en = raw_train_en.copy(deep=True)
train_tcn = raw_train_tcn.copy(deep=True)

train_en = train_en.dropna()
train_tcn = train_tcn.dropna()

train_en = train_en[
    ~train_en["product_title"].str.contains("\n") |
    ~train_en["product_title"].str.contains("\"") |
    ~train_en["product_title"].str.contains(",")
]

train_tcn = train_tcn[
    ~train_tcn["product_title"].str.contains("\n") |
    ~train_tcn["product_title"].str.contains("\"") |
    ~train_tcn["product_title"].str.contains(",")
]

train_en.loc[train_en["category"] == " Games & Collectibles\""].product_title.apply(lambda x: x[:-7])
train_en.loc[train_en["category"] == " Games & Collectibles\"", "category"] = "Toys, Games & Collectibles"

In [5]:
category_map = {
    "Game Kingdom"           : "Gaming",
    "Health & Beauty"        : "Health & Personal Care",
    "Food & Beverages"       : "Groceries",
    "Home Electronic"        : "Home Appliances",
    "Men's Bags& Accessories": "Men's Bags & Accessories",
    "Mobile & Gadgets"       : "Mobiles & Gadgets",
    "Mother & Baby"          : "Babies & Kids",
    "Hardware & 3C"          : "Laptops & Computers", # https://tw.forumosa.com/t/what-is-3c-actually/62673/2
    "Pets"                   : "Pet Care",
    "Life & Entertainment"   : "Toys, Games & Collectibles",
    "Sports & Outdoors"      : "Sports & Travel",
    "Women Bags"             : "Women's Bags",
}

train_tcn["category"] = train_tcn["category"].replace(category_map)

In [6]:
test_en = raw_test_en.copy(deep=True)
test_tcn = raw_test_tcn.copy(deep=True)

In [7]:
predict_tcn = raw_predict_tcn.copy(deep=True)

# Convert to Simplified Chinese

In [8]:
import opencc


converter = opencc.OpenCC("tw2sp.json")

train_tcn["product_title"] = train_tcn["product_title"].apply(converter.convert)
test_tcn["text"] = test_tcn["text"].apply(converter.convert)
predict_tcn["text"] = predict_tcn["text"].apply(converter.convert)

# Translate

In [9]:
import googletrans


zh2en = googletrans.Translator()


def translate(text):
    res = zh2en.translate(text, src="zh-tw", dest="en")
    return res.text

# Validation

In [10]:
test_tcn["translation_output"] = test_tcn["text"].apply(translate)

In [11]:
!wget https://storage.googleapis.com/kaggle-forum-message-attachments/932607/16344/evluation_script.py

--2020-08-01 15:12:08--  https://storage.googleapis.com/kaggle-forum-message-attachments/932607/16344/evluation_script.py
Resolving storage.googleapis.com (storage.googleapis.com)... 172.217.194.128, 2404:6800:4003:c04::80
Connecting to storage.googleapis.com (storage.googleapis.com)|172.217.194.128|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 859 [text/x-python-script]
Saving to: ‘evluation_script.py.1’

evluation_script.py 100%[===================>]     859  --.-KB/s    in 0s      

2020-08-01 15:12:09 (20.6 MB/s) - ‘evluation_script.py.1’ saved [859/859]



In [12]:
import evluation_script


bleu_score = evluation_script.eval(
    test_tcn.translation_output.tolist(),
    test_en.translation_output.tolist(),
)

print("Sacrebleu accuracy: {0:.5f}%".format(bleu_score))

Sacrebleu accuracy: 34.88294%


# Prediction

In [13]:
predict_tcn["translation_output"] = predict_tcn["text"].apply(translate)

# Save Result

In [14]:
result = predict_tcn.copy(deep=True)


result = result.drop(
    columns=[
        "text",
        "split",
    ]
)

In [15]:
result.to_csv(OUTPUT_CSV, index=False)